In [ ]:
from pathlib import Path
import pandas as pd
import sys
# Resolve project root directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    

from src.missing_value_handling_before_LSTM import (
    handle_missing_values,
    check_sampling_interval,
)

In [ ]:
TRAIN_PATH = Path(
    "/home/jetson/Desktop/ciesol-anomaly-detection/data/processed/train_features.parquet"
)

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"Train file not found: {TRAIN_PATH}"
    )

df_train = pd.read_parquet(TRAIN_PATH)
TEST_PATH = Path(
    "/home/jetson/Desktop/ciesol-anomaly-detection/data/processed/test_features.parquet"
)

if not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Train file not found: {TEST_PATH}"
    )

df_test = pd.read_parquet(TEST_PATH)
VALIDATION_PATH = Path(
    "/home/jetson/Desktop/ciesol-anomaly-detection/data/processed/validation_features.parquet"
)

if not VALIDATION_PATH.exists():
    raise FileNotFoundError(
        f"Train file not found: {VALIDATION_PATH}"
    )

df_validation = pd.read_parquet(VALIDATION_PATH)

print(f"Train dataset loaded: {df_train.shape}")
print(f"Test dataset loaded: {df_test.shape}")
print(f"Validation dataset loaded: {df_validation.shape}")

In [ ]:
check_sampling_interval(
    df_train,
    "TRAIN"
)

check_sampling_interval(
    df_validation,
    "VALIDATION"
)

check_sampling_interval(
    df_test,
    "TEST"
)

In [ ]:
MAX_INTERPOLATION_GAP_SECONDS = 60.0

df_train_final, train_nan_report = handle_missing_values(
    df_train,
    max_interpolation_gap_seconds=MAX_INTERPOLATION_GAP_SECONDS,
    raise_on_unexpected_nan=True,
    injected_anomalies=True,
)

In [ ]:
df_validation_final, validation_nan_report = handle_missing_values(
    df_validation,
    max_interpolation_gap_seconds=MAX_INTERPOLATION_GAP_SECONDS,
    raise_on_unexpected_nan=True,
    injected_anomalies=True,
)

In [ ]:
df_test_final, test_nan_report = handle_missing_values(
    df_test,
    max_interpolation_gap_seconds=MAX_INTERPOLATION_GAP_SECONDS,
    raise_on_unexpected_nan=True,
    injected_anomalies=True,
)

In [ ]:
reports = {
    "TRAIN": train_nan_report,
    "VALIDATION": validation_nan_report,
    "TEST": test_nan_report,
}

for dataset_name, report in reports.items():

    print("=" * 70)
    print(f"{dataset_name} — MISSING VALUE REPORT")
    print("=" * 70)

    print(
        "Model NaN cells:",
        report["model_nan_cells"]
    )

    print(
        "Injected missing rows:",
        report["missing_anomaly_rows"]
    )

    print("Unfilled injected NaNs:")
    print(
        report["unfilled_injected_nans"]
    )

    print("\nNon-injected sensor NaNs:")
    print(
        report["non_injected_sensor_nans"]
    )

    print()

In [ ]:
print(
    "\nany_sensor_was_missing:",
    int(df_train_final["any_sensor_was_missing"].sum())
)
print(
    "\nany_sensor_was_missing:",
    int(df_test_final["any_sensor_was_missing"].sum())
)
print(
    "\nany_sensor_was_missing:",
    int(df_validation_final["any_sensor_was_missing"].sum())
)

In [ ]:
missing_flag_columns = [
    f"{sensor}_was_missing"
    for sensor in [
        "PCO2EXT",
        "PHEXT",
        "PRAD",
        "PRGINT",
        "PTEXT",
        "PVV",
        "UVENT_cen",
        "UVENT_lN",
        "XCO2I",
        "XHINV",
        "XTINV",
        "XTS",
    ]
]

print("Missingness flags:")
print(
    df_train_final[missing_flag_columns]
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
print(
    "\nRows with any sensor missing:",
    int(df_train_final["any_sensor_was_missing"].sum())
)

In [ ]:
print("\nVALIDATION")
print(
    "any_sensor_was_missing:",
    int(
        df_validation_final["any_sensor_was_missing"].sum()
    )
)

print("\nTEST")
print(
    "any_sensor_was_missing:",
    int(
        df_test_final["any_sensor_was_missing"].sum()
    )
)

In [ ]:
print("=" * 70)
print("ANOMALY LABEL PRESERVATION")
print("=" * 70)

print("\nTRAIN:")
print(
    df_train_final["tipo_anomalia"]
    .value_counts(dropna=False)
)

In [ ]:
print(
    "Train labels preserved:",
    train_nan_report["labels_preserved"]
)

In [ ]:
derived_features = [
    "delta_temp",
    "delta_hum",
    "co2_gradient",
]

print("\nTRAIN — derived feature NaNs:")
print(
    df_train_final[derived_features]
    .isna()
    .sum()
)

print("\nVALIDATION — derived feature NaNs:")
print(
    df_validation_final[derived_features]
    .isna()
    .sum()
)

print("\nTEST — derived feature NaNs:")
print(
    df_test_final[derived_features]
    .isna()
    .sum()
)

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_train_final.to_parquet(
    PROCESSED_DIR / "train_features_imputed_NaN_before_LSTM.parquet",
    index=False,
)

df_validation_final.to_parquet(
    PROCESSED_DIR / "validation_features_imputed_NaN_before_LSTM.parquet",
    index=False,
)

df_test_final.to_parquet(
    PROCESSED_DIR / "test_features_imputed_NaN_before_LSTM.parquet",
    index=False,
)

In [ ]:
df_test_final.columns